# Step 4: Compare groups and explain the differences
**Questions:** Do theme types differ in accessibility? Do big brands differ from smaller merchants?
Does that still hold once we account for store size and category (regression)?

One-time setup in Terminal: `pip3 install pandas statsmodels matplotlib jupyter`
then open this notebook with `jupyter notebook analysis/step4_compare.ipynb` (or in VS Code).

This notebook reads the same clean tables you uploaded to BigQuery (from `data/tables/`),
and uses the **same alt-text rules as the SQL view in Step 3**, so numbers match.


In [1]:
import re, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

BASE = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "analysis" else os.getcwd()
T = lambda name: pd.read_csv(os.path.join(BASE, "data", "tables", f"{name}.csv"), keep_default_na=False, na_values=[""])
CHARTS = os.path.join(BASE, "analysis", "charts"); os.makedirs(CHARTS, exist_ok=True)

stores, images, pages, issues = T("stores"), T("images_alt"), T("pages"), T("page_issues")
stores = stores[stores.status == "ok"]
print(f"{len(stores)} stores | {len(images):,} sampled images | {len(pages)} pages checked | {len(issues)} page issues")

# Chart style: thin marks, recessive grid, one accent colour (validated reference palette)
BLUE, ORANGE, AQUA = "#008060", "#eb6834", "#1baf7a"
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e0"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": GRID,
                     "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.titlecolor": INK, "axes.titlesize": 12, "font.size": 10,
                     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True})


Matplotlib is building the font cache; this may take a moment.


422 stores | 30,661 sampled images | 838 pages checked | 2765 page issues


## 4.1 Classify alt text (same rules as the Step 3 SQL view)


In [3]:
def alt_class(alt, title):
    a = (alt if isinstance(alt, str) else "").strip(); al = a.lower()
    tl = (title if isinstance(title, str) else "").lower().strip()
    if a == "":
        return "missing"
    if (re.search(r"\.(jpe?g|png|webp|gif|avif)$", al)
            or re.search(r"(::|\|\||option\[|^img[_-]?\d|^dsc|pdp\d?|_main\b|^colou?r:|^size:|#colou?r_)", al)
            or re.search(r"^(featured|slideshow|image|img|photo|picture|thumbnail|banner|default|product)( ?\d+)?$", al)
            or not re.search(r"[A-Za-z]{3}", a)
            or (not re.search(r"\s", a) and re.search(r"[-_]", a))):
        return "junk"
    if al == tl or len(re.findall(r"[A-Za-z]+", a)) <= 3 or al in tl:
        return "minimal"
    return "descriptive"

images["alt_class"] = [alt_class(a, t) for a, t in zip(images.alt, images.product_title)]
images.alt_class.value_counts(normalize=True).round(3)


alt_class
descriptive    0.421
missing        0.370
minimal        0.112
junk           0.097
Name: proportion, dtype: float64

## 4.2 Build one row per store (the unit of analysis)


In [ ]:
per_store_alt = (images.assign(missing=images.alt_class.eq("missing"),
                               unusable=images.alt_class.isin(["missing", "junk"]))
                 .groupby("domain").agg(n_images=("alt_class", "size"),
                                        pct_missing=("missing", "mean"),
                                        pct_unusable=("unusable", "mean")))
prod_pages = pages[pages.page_type == "product"].set_index("domain")
contrast_fail = set(issues.query("page_type == 'product' and rule_id == 'color-contrast'").domain)

df = (stores.set_index("domain")[["category", "sample_group", "theme_family", "base_theme", "n_products"]]
      .join(per_store_alt, how="inner")
      .join(prod_pages[["n_rule_violations"]], how="left"))
df["contrast_fail"] = df.index.isin(contrast_fail).astype(int)
df["pct_missing"] *= 100; df["pct_unusable"] *= 100
df = df[df.theme_family != "unknown"]
print(len(df), "stores in the analysis"); df.head()


## 4.3 Compare groups: mean % missing with a 95% confidence interval


In [ ]:
def summarise(g, col="pct_missing"):
    n = len(g); m = g[col].mean(); se = g[col].std(ddof=1) / np.sqrt(n) if n > 1 else np.nan
    return pd.Series({"n_stores": n, "mean": round(m, 1), "ci_low": round(m - 1.96 * se, 1),
                      "ci_high": round(m + 1.96 * se, 1),
                      "pct_contrast_fail": round(100 * g.contrast_fail.mean(), 1)})

by_theme = df.groupby("theme_family").apply(summarise, include_groups=False).sort_values("mean")
by_group = df.groupby("sample_group").apply(summarise, include_groups=False)
display(by_theme); display(by_group)


**How to read this:** if two groups' confidence intervals overlap a lot, the data can't
tell them apart yet. Small groups (few stores) have wide intervals, so say so.


## 4.4 Charts


In [ ]:
# Chart 1: how alt text breaks down (one series, so no legend; labels on the bars)
order = ["descriptive", "minimal", "junk", "missing"]
shares = images.alt_class.value_counts(normalize=True).reindex(order) * 100
fig, ax = plt.subplots(figsize=(7, 2.8))
# same colours as the video slides: good -> bad
CAT = {"descriptive": "#008060", "minimal": "#95BF47", "junk": "#d9822b", "missing": "#c0392b"}
ax.barh([o.capitalize() for o in order[::-1]], shares[order[::-1]], color=[CAT[o] for o in order[::-1]], height=0.55)
for y, v in enumerate(shares[order[::-1]]):
    ax.text(v + 0.8, y, f"{v:.0f}%", va="center", color=INK, fontweight="bold")
ax.set_xlabel("% of sampled product images"); ax.grid(axis="y", visible=False)
ax.set_title("Most product images don't have a useful description", loc="left")
fig.tight_layout(); fig.savefig(os.path.join(CHARTS, "1_alt_text_breakdown.png"), dpi=200); plt.show()


In [ ]:
# Chart 2: % missing by theme family, with 95% CI error bars (same style as the video slide)
NAMES = {"off_the_shelf": "Off-the-shelf theme", "off_the_shelf_customised": "Off-the-shelf, customised",
         "bespoke": "Fully custom (bespoke)"}
ORDER = ["off_the_shelf", "bespoke", "off_the_shelf_customised"]          # top to bottom, as on the slide
t = by_theme.reset_index().set_index("theme_family").reindex(ORDER).dropna(subset=["mean"]).reset_index()[::-1]
labels = [f"{NAMES.get(f, f)}\n(n={int(n)})" for f, n in zip(t.theme_family, t.n_stores)]
fig, ax = plt.subplots(figsize=(7.5, 3.2))
ax.barh(labels, t["mean"], xerr=[t["mean"] - t.ci_low, t.ci_high - t["mean"]],
        color=BLUE, height=0.55, error_kw={"ecolor": "#004c3f", "elinewidth": 1.5, "capsize": 4})
for y, v in enumerate(t["mean"]):
    ax.text(1.5, y, f"{v:.0f}%", va="center", color="white", fontweight="bold")
ax.set_xlim(0, 65); ax.set_xticks([0, 20, 40, 60]); ax.set_xticklabels(["0%", "20%", "40%", "60%"])
ax.set_xlabel("Average % of a store's images with no description (95% CI)"); ax.grid(axis="y", visible=False)
ax.set_title("Missing descriptions are about the same on every theme type", loc="left")
fig.tight_layout(); fig.savefig(os.path.join(CHARTS, "2_missing_by_theme.png"), dpi=200); plt.show()


In [ ]:
# Chart 3: most common page failures (product pages)
n_prod = (pages.page_type == "product").sum()
top = (issues[issues.page_type == "product"].groupby("rule_id").domain.nunique()
       .div(n_prod).mul(100).sort_values().tail(8))
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.barh(top.index, top.values, color=BLUE, height=0.55)
for y, v in enumerate(top.values):
    ax.text(v + 0.8, y, f"{v:.0f}%", va="center", color=INK)
ax.set_xlabel(f"% of product pages failing (n={n_prod})"); ax.grid(axis="y", visible=False)
ax.set_title("Most common accessibility failures on product pages", loc="left")
fig.tight_layout(); fig.savefig(os.path.join(CHARTS, "3_top_page_failures.png"), dpi=200); plt.show()


## 4.5 Regression: does theme type still matter after accounting for other factors?
Groups can differ for other reasons (e.g. big brands have bigger catalogs AND custom themes).
Regression estimates each factor's effect **holding the others constant**.
- Outcome: `pct_missing` (0-100)
- Factors: theme family, sample group, store size (log of product count), category
- `cov_type="HC1"` = robust standard errors (don't assume equal spread across stores)


In [ ]:
formula = "pct_missing ~ C(theme_family, Treatment('bespoke')) + np.log1p(n_products) + C(category)"
if df.sample_group.nunique() > 1:
    formula += " + C(sample_group, Treatment('big_brand'))"
model = smf.ols(formula, data=df).fit(cov_type="HC1")
coefs = model.summary2().tables[1]
coefs[~coefs.index.str.startswith("C(category")].round(3)   # category effects hidden to keep it readable


**How to read this:** each coefficient = change in % of images missing alt text compared
with the baseline group (bespoke themes / big brands), with everything else held equal.
A p-value below 0.05 means the difference is unlikely to be chance.
Write down: R-squared (how much the model explains) and which effects are clearly non-zero.


In [ ]:
# Same idea for a yes/no outcome: does the product page fail colour contrast? (logistic regression)
logit_formula = formula.replace("pct_missing", "contrast_fail").replace(" + C(category)", "")
logit = smf.logit(logit_formula, data=df).fit(disp=0)
np.exp(logit.params).round(2).to_frame("odds_ratio")
